Working with the public 14-year IceCube point-source data
==

This tutorial shows how to use the IceCube public 14-year point-source data with SkyLLH.

In [1]:
import numpy as np
from matplotlib import pyplot as plt
import scipy.stats
import sys
import seaborn as sns

sns.set_theme(
    context="notebook",
    style="darkgrid",
    palette="deep",
    font="sans-serif",
)

sys.path.append("/Users/amithan/skyllh_14/skyllh")

Creating a configuration
---

First we have to create a configuration instance.

In [2]:
from skyllh.core.config import Config
cfg = Config()

Getting the datasets
---

Importing dataset definition of the public 14-year point-source data set:

In [3]:
from skyllh.datasets.i3.PublicData_14y_ps import create_dataset_collection

The collection of datasets can be created using the ``create_dataset_collection`` function. This function requires the base path to the data repository.   

``dataset_names`` property provides a list of all the data sets defined in the data set collection of the public point-source data.   

The individual data sets ``IC86_I``, ``IC86_II``, ``IC86_III``, ``IC86_IV``, ``IC86_V``, ``IC86_VI``, ``IC86_VII``, ``IC86_VIII``, ``IC86_IX``, ``IC86_X``, and ``IC86_XI`` are also available as a single combined data set ``IC86_I-IX``, because these data sets share the same detector simulation and event selection. Hence, we can get a list of data sets via the access operator ``[dataset1, dataset2, ...]`` of the ``dsc`` instance:

In [4]:
dsc = create_dataset_collection(
    cfg=cfg, 
    base_path='/Users/amithan/Downloads/',
    sub_path_fmt='PublicData_14y')
print(dsc.dataset_names)
datasets = dsc['IC40', 'IC59', 'IC79', 'IC86_I-XI']

['IC40', 'IC59', 'IC79', 'IC86_I', 'IC86_I-XI', 'IC86_II', 'IC86_III', 'IC86_IV', 'IC86_IX', 'IC86_V', 'IC86_VI', 'IC86_VII', 'IC86_VIII', 'IC86_X', 'IC86_XI']


Getting the analysis
---

The analysis used for the published PRL results is referred in SkyLLH as "*traditional point-source analysis*" and is pre-defined:

In [5]:
from skyllh.core.source_model import PointLikeSource
from skyllh.analyses.i3.publicdata_ps.time_integrated_ps import create_analysis

In [6]:
help(create_analysis)

Help on function create_analysis in module skyllh.analyses.i3.publicdata_ps.time_integrated_ps:

create_analysis(cfg, datasets, source, refplflux_Phi0=1, refplflux_E0=1000.0, refplflux_gamma=2.0, refplflux_Ec=inf, ns_seed=10.0, ns_min=0.0, ns_max=1000.0, gamma_seed=3.0, gamma_min=1.0, gamma_max=4.0, kde_smoothing=False, minimizer_impl='LBFGS', minimizer_max_rep=100, cap_ratio=False, compress_data=False, keep_data_fields=None, evt_sel_delta_angle_deg=10, construct_sig_generator=True, tl=None, ppbar=None, logger_name=None)
    Creates the Analysis instance for this particular analysis.
    
    Parameters
    ----------
    cfg : instance of Config
        The instance of Config holding the local configuration.
    datasets : list of Dataset instances
        The list of Dataset instances, which should be used in the
        analysis.
    source : PointLikeSource instance
        The PointLikeSource instance defining the point source position.
    refplflux_Phi0 : float
        The flux 

Defining the Source: Here we use NGC 1068 as the source

Once the source is defined the ``create_analysis`` function can create the analysis instance

In [7]:
src_ra = 40.67  # degrees
src_dec = -0.01  # degrees

source = PointLikeSource(
    ra=np.radians(src_ra),
    dec=np.radians(src_dec)
)

ana = create_analysis(
    cfg=cfg,
    datasets=datasets,
    source=source
)

100%|██████████| 136/136 [00:00<00:00, 272.84it/s]


Initializing a trial
---

After the `Analysis` instance was created trials can be run. To do so the analysis needs to be initialized with some trial data. For instance we could initialize the analysis with the experimental data to "unblind" the analysis afterwards. Technically the `TrialDataManager` of each log-likelihood ratio function, i.e. dataset, is initialized with data.

The `Analysis` class provides the method `initialize_trial` to initialize a trial with data. It takes a list of `DataFieldRecordArray` instances holding the events. If we want to initialize a trial with the experimental data, we can get that list from the `Analysis` instance itself:

In [8]:
events_list = [ data.exp for data in ana.data_list ]
ana.initialize_trial(events_list)

Maximizing the log-likelihood ratio function
---

After initializing a trial, we can maximize the LLH ratio function using the `maximize_llhratio` method of the `Analysis` class. This method requires a ``RandomStateService`` instance in case the minimizer does not succeed and a new set of initial values for the fit parameters need to get generated. The method returns a 4-element tuple. The first element is the set of fit parameters used in the maximization. The second element is the value of the LLH ration function at its maximum. The third element is the array of the fit parameter values at the maximum, and the forth element is the status dictionary of the minimizer.

In [9]:
from skyllh.core.random import RandomStateService
rss = RandomStateService(seed=1)

In [10]:
(log_lambda_max, fitparam_values, status) = ana.llhratio.maximize(rss)

In [11]:
print(f'log_lambda_max = {log_lambda_max}')
print(f'fitparam_values = {fitparam_values}')
print(f'status = {status}')

log_lambda_max = 14.576758408087755
fitparam_values = [80.09381868  3.20884803]
status = {'grad': array([-3.62369895e-06, -2.31360821e-04]), 'task': 'CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH', 'funcalls': 13, 'nit': 10, 'warnflag': 0, 'skyllh_minimizer_n_reps': 0, 'n_llhratio_func_calls': 13}


Calculating the test-statistic
---

Using the maximum of the LLH ratio function and the fit parameter values at the maximum we can calculate the test-statistic using the `calculate_test_statistic` method of the `Analysis` class:

In [12]:
TS = ana.calculate_test_statistic(log_lambda_max, fitparam_values)
print(f'TS = {TS:.3f}')

TS = 29.154


## Unblinding the data

After creating the analysis instance we can unblind the data for the choosen source. Hence, we initialize the analysis with a trial of the experimental data, maximize the log-likelihood ratio function for all given experimental data events, and calculate the test-statistic value. The analysis instance has the method ``unblind`` that can be used for that. This method requires a ``RandomStateService`` instance in case the minimizer does not succeed and a new set of initial values for the fit parameters need to get generated.

In [13]:
help(ana.unblind)

Help on method unblind in module skyllh.core.analysis:

unblind(minimizer_rss, tl=None) method of skyllh.core.analysis.SingleSourceMultiDatasetLLHRatioAnalysis instance
    Evaluates the unscrambled data, i.e. unblinds the data.
    
    Parameters
    ----------
    minimizer_rss : instance of RandomStateService
        The instance of RandomStateService that should be used by the
        minimizer to generate new random initial fit parameter values.
    tl : instance of TimeLord | None
        The optional instance of TimeLord that should be used to time the
        maximization of the LLH ratio function.
    
    Returns
    -------
    TS : float
        The test-statistic value.
    global_params_dict : dict
        The dictionary holding the global parameter names and their
        best fit values. It includes fixed and floating parameters.
    status : dict
        The status dictionary with information about the performed
        minimization process of the negative of the log-

In [14]:
from skyllh.core.random import RandomStateService
rss = RandomStateService(seed=1)

(ts, x, status) = ana.unblind(rss)

In [15]:
print(f'TS = {ts:.3f}')
print(f'ns = {x["ns"]:.2f}')
print(f'gamma = {x["gamma"]:.2f}')

TS = 29.154
ns = 80.09
gamma = 3.21


## Calculating the corresponding flux normalization 

By default the analysis is created with a flux normalization of 1 $\text{GeV}^{-1}\text{s}^{-1}\text{cm}^{-2}\text{sr}^{-1}$ (see `refplflux_Phi0` argument of the `create_analysis` method). The analysis instance has the method `calculate_fluxmodel_scaling_factor` that calculates the scaling factor the reference flux normalization has to be multiplied with to represent a given analysis result, i.e. $n_{\text{s}}$ and $\gamma$ value. This function takes the detected mean $n_{\text{s}}$ value as first argument and the list of source parameter values as second argument:

In [16]:
scaling_factor = ana.calculate_fluxmodel_scaling_factor(x['ns'], [x['ns'], x['gamma']])
print(f'Flux scaling factor = {scaling_factor:.3e}')

Flux scaling factor = 2.869e-14


Hence, our result corresponds to a power-law flux of:

In [17]:
print(f'{scaling_factor:.3e}'' (E/1000 GeV)^{-'f'{x["gamma"]:.2f}'+'} 1/(GeV s cm^2 sr)')

2.869e-14 (E/1000 GeV)^{-3.21} 1/(GeV s cm^2 sr)
